# 🎨 Image Style Transfer - Google Colab Setup

This notebook trains an AdaIN-based Arbitrary Style Transfer model using GPU.

**Runtime:** Runtime > Change runtime type > **GPU**

---

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/

# Create project folder if not exists
!mkdir -p style_transfer
%cd style_transfer
!pwd

## Step 2: Upload Project Files

Upload these files to the `style_transfer` folder in Google Drive:
- `train.py`
- `test.py`
- `models/adain_model.py`
- `models/__init__.py`
- `losses/losses.py`
- `losses/__init__.py`
- `datasets/datasets.py`
- `datasets/__init__.py`

Or clone from GitHub:

In [ ]:
# Option A: Clone from GitHub (if you have one)
# !git clone https://github.com/YOUR_USERNAME/style-transfer.git

# Option B: Upload manually
# Files should be in: /content/drive/MyDrive/style_transfer/

# Verify files exist
!ls -la

## Step 3: Install Dependencies

In [ ]:
!pip install torch torchvision scikit-image pillow tqdm -q
print("Dependencies installed!")

## Step 4: Download Datasets

We'll download MSCOCO validation set (5K images) and style images.

In [ ]:
# Create dataset folders
!mkdir -p datasets/content datasets/style

# Download first 1000 images from MSCOCO (faster for testing)
!wget -q "https://raw.githubusercontent.com/gupta-abhay/COCO-dataset/master/data/train2017.zip" -O train2017.zip 2>/dev/null || echo "Downloading alternative..."

# Alternative: Download from Google Drive (upload val2017.zip to Drive)
# !gdown --folder YOUR_FOLDER_ID

# Use sample images from web for testing
!wget -q "https://raw.githubusercontent.com/SharanMotta/NeuralStyleTransfer/main/images/content.jpg" -O datasets/content/test.jpg 2>/dev/null
!wget -q "https://raw.githubusercontent.com/SharanMotta/NeuralStyleTransfer/main/images/style.jpg" -O datasets/style/style.jpg 2>/dev/null

print("Datasets downloaded!")
!ls datasets/

## Step 5: Verify GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Step 6: Training

Training with GPU - should be much faster!

In [ ]:
!python train.py \
  --content-dir ./datasets/content \
  --style-dir ./datasets/style \
  --output-dir ./checkpoints \
  --epochs 2 \
  --batch-size 16 \
  --image-size 256 \
  --log-interval 10

## Step 7: Testing

In [ ]:
# Check if model was saved
!ls -la checkpoints/

In [ ]:
!python test.py \
  --model ./checkpoints/final_model.pth \
  --content ./datasets/content/test.jpg \
  --style ./datasets/style/style.jpg \
  --output ./output.jpg

## Step 8: Display Results

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

content = Image.open('./datasets/content/test.jpg')
style = Image.open('./datasets/style/style.jpg')
output = Image.open('./output.jpg')

axes[0].imshow(content)
axes[0].set_title('Content')
axes[0].axis('off')

axes[1].imshow(style)
axes[1].set_title('Style')
axes[1].axis('off')

axes[2].imshow(output)
axes[2].set_title('Stylized Output')
axes[2].axis('off')

plt.tight_layout()
plt.show()